# Financial analyst: package client

This notebook calls the implementation in `src/sec_analyst/`. It does not contain a second model. Select the project `.venv` kernel after installing the editable package. Run from the repository root or this `notebooks/` folder. The supplied parent notebook remains the reference.

The prepared dataset must already exist. No downloads or data conversion happen here. Proposed financial values are for human review.


In [ ]:
from pathlib import Path
import yaml
import pandas as pd
from sec_analyst.warehouse import connect_warehouse
from sec_analyst.concepts import register_concepts
from sec_analyst.checks import IDENTITIES, identity_check
from sec_analyst.mapper_data import prepare_line_items
from sec_analyst.custom_map import prepare_custom_map
from sec_analyst.service import QuestionService

root = Path.cwd().resolve()
if root.name == "notebooks":
    root = root.parent
settings = yaml.safe_load((root / "config/config.yaml").read_text())
processed = root / settings["processed_dir"]
print("Project:", root)
print("Quarters:", settings["quarters"])


## Inspect the prepared warehouse
DuckDB reads our Parquet files. The concept table connects analyst language to the notebook's ordered financial tags.


In [ ]:
with connect_warehouse(processed, settings["quarters"]) as connection:
    register_concepts(connection)
    print(connection.execute("SELECT concept, count(*) AS tags FROM kb GROUP BY concept ORDER BY concept").df().to_string(index=False))


## Check accounting relationships
These are the package functions, using the original notebook's tolerance. A relationship failing this tolerance does not by itself establish a bad filing.


In [ ]:
with connect_warehouse(processed, settings["quarters"]) as connection:
    results = [(name, *identity_check(connection, *spec)) for name, spec in IDENTITIES.items()]
print(pd.DataFrame(results, columns=["relationship", "contexts", "holds"]).to_string(index=False))


## Ask the existing model
This is the same parser, mapping and SQL path used by the API. Verification reruns SQL; it cannot establish that the selected tag means what the user intended.


In [ ]:
question = "What was Birner Dental Management Services's revenue in fiscal 2015?"
with connect_warehouse(processed, settings["quarters"]) as connection:
    service = QuestionService(connection)
    parsed = service.parser.parse(question)
    prepare_line_items(connection, processed)
    prepare_custom_map(connection, parsed.get("ciks", []))
    candidate = service.answer(question, 4)
    print("Candidate:", candidate)
    if candidate["status"] == "answered":
        print("SQL verification:", service.verify(candidate["sql"], candidate["value"]))
    print("Decision: refer for human review")
    service.run_sql.cache_clear()


## Repeat the evaluations

Run these from the repository root terminal, using the same environment:

```bash
python3 -m sec_analyst.data_checks
python3 -m sec_analyst.evaluate_mapper
python3 -m sec_analyst.evaluate_answers
python3 -m sec_analyst.evaluate_temporal
```

See `docs/rd1-checklist.md` and `docs/evidence/` for measured results and limitations. The quarterly experiment preserves the notebook diagnostic and separately restricts the training inputs by time; it does not replace the given model.
